In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import os
import joblib
from pathlib import Path
import sys
from sklearn.metrics import mean_squared_error, r2_score
PROJECT_ROOT = Path.cwd().parents[0]

In [2]:
import pandas as pd

results = []

for h in [1, 2, 3, 4]:
   
    # Load files (adjust filenames as needed)
    gpf = pd.read_csv(Path(PROJECT_ROOT / "outputs"/ "feature_importance" / "standard" / f"GPFI_W_{h}_standard.csv"))
    groups = pd.read_csv(Path(PROJECT_ROOT / "outputs"/ "feature_importance" / "standard" / f"Groups_W_{h}_standard.csv"))
   
    # Make sure column names match
    gpf = gpf.rename(columns={"Group": "Group", "Mean_Importance": "Importance"})
    groups = groups.rename(columns={"Group": "Group", "Feature": "Feature"})
   
    # ----------------------------
    # Get top 3 groups
    # ----------------------------
    top_groups = gpf.sort_values("Importance", ascending=False).head(3).copy()
   
    # Add rank
    top_groups["Rank"] = [1, 2, 3]
   
    # ----------------------------
    # Merge with feature list
    # ----------------------------
    merged = groups.merge(top_groups, on="Group")
   
    # ----------------------------
    # Aggregate features per group
    # ----------------------------
    summary = (
        merged.groupby(["Group", "Importance", "Rank"], as_index=False)
        .agg({
            "Feature": lambda x: ", ".join(sorted(x)),
            "Feature": "count"
        })
    )
   
    # Rename columns
    summary = summary.rename(columns={"Feature": "Group_Size"})
   
    # Add features separately
    features = (
        merged.groupby("Group")["Feature"]
        .apply(lambda x: ", ".join(sorted(x)))
        .reset_index()
    )
   
    summary = summary.merge(features, on="Group")
   
    # Add horizon
    summary["Horizon"] = h
   
    results.append(summary)

final_df = pd.concat(results, ignore_index=True)

# Reorder columns
final_df = final_df[["Horizon", "Rank", "Group", "Importance", "Group_Size", "Feature"]]

print(final_df)

    Horizon  Rank  Group  Importance  Group_Size  \
0         1     3    108    0.064932           5   
1         1     1    184    0.155484          17   
2         1     2    230    0.076548           5   
3         2     3    172    0.010713           2   
4         2     1    380    0.016214          17   
5         2     2    441    0.013931           6   
6         3     2    223    0.010532           5   
7         3     3    251    0.009388           5   
8         3     1    287    0.010617           5   
9         4     3    177    0.009395           6   
10        4     2    200    0.009669          10   
11        4     1    234    0.010662           4   

                                              Feature  
0   h_500_weekave_pacific_ocean_pc6, h_850_weekave...  
1   W_current_week, h_500_weekave_conus_pc1, h_500...  
2   h_500_weekave_pacific_ocean_pc3, h_850_weekave...  
3   h_850_weekave_pacific_trop_pc14, slp_weekave_p...  
4   W_current_week, h_500_weekave_conus_pc1

In [4]:
# Sort nicely
final_df = final_df.sort_values(["Horizon", "Rank"]).copy()

# Round importance
final_df["Importance"] = final_df["Importance"].round(3)

# Optional: rename columns for table
latex_df = final_df.rename(columns={
    "Horizon": "Lead time",
    "Rank": "Rank",
    "Group": "Group",
    "Importance": "Importance",
    "Group_Size": "Group size",
    "Feature": "Features"
})

# Escape underscores for LaTeX
latex_df["Features"] = latex_df["Features"].str.replace("_", r"\_", regex=False)

latex_table = latex_df.to_latex(
    index=False,
    escape=False,   # keep \_ intact
    longtable=True, # good for long feature lists
    column_format=r"ccrlp{0.45\textwidth",
    caption="Top three most important grouped PFI predictor groups for RF forecasts in the West region across lead times.",
    label="tab:west_top_groups"
)

with open("west_top_groups_table.tex", "w") as f:
    f.write(latex_table)

print(latex_table)


\begin{longtable}{ccrlp{0.45\textwidth}
\caption{Top three most important grouped PFI predictor groups for RF forecasts in the West region across lead times.} \label{tab:west_top_groups} \\
\toprule
Lead time & Rank & Group & Importance & Group size & Features \\
\midrule
\endfirsthead
\caption[]{Top three most important grouped PFI predictor groups for RF forecasts in the West region across lead times.} \\
\toprule
Lead time & Rank & Group & Importance & Group size & Features \\
\midrule
\endhead
\midrule
\multicolumn{6}{r}{Continued on next page} \\
\midrule
\endfoot
\bottomrule
\endlastfoot
1 & 1 & 184 & 0.155000 & 17 & W\_current\_week, h\_500\_weekave\_conus\_pc1, h\_500\_weekave\_mexico\_gulf\_pc2, h\_500\_weekave\_southern\_canada\_pc5, h\_850\_weekave\_conus\_pc2, h\_850\_weekave\_mexico\_gulf\_pc2, h\_850\_weekave\_southern\_canada\_pc5, slp\_weekave\_conus\_pc4, slp\_weekave\_mexico\_gulf\_pc2, slp\_weekave\_southern\_canada\_pc6, t\_500\_weekave\_conus\_pc2, t\_500\_weekave\

In [2]:
from pathlib import Path
import pandas as pd
import re

BASE_DIR = Path("/home/mfholth/subseasonal/weekly_data/final_code/outputs/tuning/")  # change this
OUT_DIR = Path("/home/mfholth/subseasonal/weekly_data/final_code/notebooks/")           # change this
OUT_DIR.mkdir(parents=True, exist_ok=True)

REGIONS = ["MW", "NE", "SE", "SW", "W"]

def infer_region(filename):
    for r in REGIONS:
        if re.search(fr"(^|[_-]){r}([_-]|\.|$)", filename):
            return r
    raise ValueError(f"Could not infer region from filename: {filename}")

def load_tuning_folder(folder_name):
    folder = BASE_DIR / folder_name
    dfs = []

    for file in sorted(folder.glob("*.csv")):
        df = pd.read_csv(file)
        df["Region"] = infer_region(file.name)
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)

    # Replace blank/NA max_features with default p/3
    if "max_features" in out.columns:
        out["max_features"] = (
            out["max_features"]
            .replace(r"^\s*$", pd.NA, regex=True)
            .fillna("p/3")
        )

    # Put Region first if desired
    cols = ["Region"] + [c for c in out.columns if c != "Region"]
    out = out[cols]

    return out

def make_latex_table(df, caption, label, filename):
    latex_df = df.copy()

    # Escape underscores in string columns
    for col in latex_df.select_dtypes(include="object").columns:
        latex_df[col] = latex_df[col].astype(str).str.replace("_", r"\_", regex=False)

    latex = latex_df.to_latex(
        index=False,
        escape=False,
        longtable=False,
        caption=caption,
        label=label,
        float_format="%.3f"
    )

    out_path = OUT_DIR / filename
    out_path.write_text(latex)
    return latex

# Load folders
standard_df = load_tuning_folder("standard")
weighted_df = load_tuning_folder("weighted")

# Save combined CSVs
standard_df.to_csv(OUT_DIR / "rf_tuning_standard_combined.csv", index=False)
weighted_df.to_csv(OUT_DIR / "rf_tuning_weighted_combined.csv", index=False)

# Save LaTeX tables
make_latex_table(
    standard_df,
    caption="Selected RF hyperparameters from standard RMSE-based tuning.",
    label="tab:rf_tuning_standard",
    filename="rf_tuning_standard.tex"
)

make_latex_table(
    weighted_df,
    caption="Selected RF hyperparameters from weighted RMSE-based tuning.",
    label="tab:rf_tuning_weighted",
    filename="rf_tuning_weighted.tex"
)

'\\begin{table}\n\\caption{Selected RF hyperparameters from weighted RMSE-based tuning.}\n\\label{tab:rf_tuning_weighted}\n\\begin{tabular}{lrrllrlrrr}\n\\toprule\nRegion & Horizon & Best_Score & region & mode & max_depth & max_features & min_samples_leaf & min_samples_split & n_estimators \\\\\n\\midrule\nMW & 1 & 2.943 & MW & weighted & 20 & p/3 & 1 & 10 & 400 \\\\\nMW & 2 & 3.367 & MW & weighted & 20 & p/3 & 1 & 5 & 200 \\\\\nMW & 3 & 3.428 & MW & weighted & 10 & log2 & 1 & 10 & 300 \\\\\nMW & 4 & 3.441 & MW & weighted & 20 & sqrt & 1 & 5 & 200 \\\\\nNE & 1 & 2.738 & NE & weighted & 20 & p/3 & 4 & 2 & 400 \\\\\nNE & 2 & 3.077 & NE & weighted & 10 & p/3 & 4 & 2 & 200 \\\\\nNE & 3 & 3.126 & NE & weighted & 50 & sqrt & 2 & 5 & 400 \\\\\nNE & 4 & 3.143 & NE & weighted & 20 & sqrt & 1 & 5 & 300 \\\\\nSE & 1 & 2.397 & SE & weighted & 20 & p/3 & 4 & 10 & 400 \\\\\nSE & 2 & 2.713 & SE & weighted & 20 & sqrt & 4 & 10 & 300 \\\\\nSE & 3 & 2.738 & SE & weighted & 50 & log2 & 4 & 10 & 300 \\\\\

Bootstrap results

In [35]:
sys.path.append(str(PROJECT_ROOT / "src"))


,Date,W,SW,MW,SE,NE
0,202101,274.615378,279.611592,271.973366,281.417534,270.990314
1,202102,274.318620,277.382252,272.174428,277.069593,269.981012
2,202103,274.965657,280.886398,271.035402,279.680089,270.628170
3,202104,270.837034,279.743442,266.311598,282.009834,266.857439
4,202105,274.971182,282.714832,270.436346,277.661250,265.332271


In [80]:
RF_preds = pd.read_csv(Path(PROJECT_ROOT / "outputs"/ "predictions_2021_2024_RF" / f"RF_preds_standard_2021_2024.csv"))
# create forecast date
year = RF_preds["Reference_date"] // 100
week = RF_preds["Reference_date"] % 100

offset = (week - 1) + RF_preds["Horizon"]

RF_preds["Forecast_date"] = (
    (year + offset // 52) * 100
    + (offset % 52) + 1
).astype(int)

RF_preds.head(20)
RF_preds["RF_SE"] = (
    RF_preds["Pred_detrended"] -
    RF_preds["Obs_detrended"]
)**2

RF_preds["Clim_SE"] = (
    RF_preds["Obs_detrended"]
)**2

# positive = EESN better than climatology
RF_preds["d_t"] = (
    RF_preds["Clim_SE"] -
    RF_preds["RF_SE"]
)
paired = RF_preds
paired.head()

,Reference_date,Forecast_date,Observed,Predicted,Obs_detrended,Pred_detrended,Horizon,Region,Z_score,RF_SE,Clim_SE,d_t
0,202106,202107,268.648148,272.705561,-5.898702,-1.841288,1,W,-1.316462,16.462607,34.794689,18.332082
1,202107,202108,273.754786,274.447271,-1.278074,-0.585589,1,W,0.235794,0.479536,1.633474,1.153938
2,202108,202109,274.978508,275.498767,-0.613441,-0.093181,1,W,0.000021,0.270670,0.376310,0.105640
3,202109,202110,277.162459,275.431137,0.925956,-0.805366,1,W,1.345645,2.997476,0.857394,-2.140081
4,202110,202111,276.235142,276.526807,-0.737544,-0.445880,1,W,-0.376554,0.085068,0.543972,0.458903


In [85]:
eesn = pd.read_csv(Path(PROJECT_ROOT / "notebooks"/ "test_pred_obs.csv"))
eesn["EESN_SE"] = (
    eesn["Pred_detrended"] -
    eesn["Obs_detrended"]
)**2

eesn["Clim_SE"] = (
    eesn["Obs_detrended"]
)**2

# positive = EESN better than climatology
eesn["d_t"] = (
    eesn["Clim_SE"] -
    eesn["EESN_SE"]
)
paired = eesn

In [86]:
import numpy as np
import pandas as pd

def block_bootstrap_mean_difference(
    x,
    block_length=8,
    n_boot=5000,
    seed=42
):
    """
    Circular block bootstrap for the mean paired loss difference.

    Parameters
    ----------
    x : array-like
        Time-ordered paired differences d_t.
        Positive values indicate RF outperforms climatology.
    block_length : int
        Number of consecutive weeks per resampled block.
    n_boot : int
        Number of bootstrap replicates.
    seed : int
        Random seed.

    Returns
    -------
    boot_means : np.ndarray
        Bootstrap distribution of mean(d_t).
    """
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]

    n = len(x)
    rng = np.random.default_rng(seed)

    boot_means = np.empty(n_boot)

    for b in range(n_boot):
        sampled_indices = []

        while len(sampled_indices) < n:
            start = rng.integers(0, n)

            block = (
                start + np.arange(block_length)
            ) % n

            sampled_indices.extend(block)

        sampled_indices = np.asarray(sampled_indices[:n])

        boot_means[b] = x[sampled_indices].mean()

    return boot_means

In [81]:
sub = (
    paired[
        (paired["Region"] == "W") &
        (paired["Horizon"] == 4)
    ]
    .sort_values("Forecast_date")
    .dropna(subset=["d_t"])
    .copy()
)

d = sub["d_t"].to_numpy()

boot_means = block_bootstrap_mean_difference(
    d,
    block_length=8,
    n_boot=5000,
    seed=42
)

observed_mean = np.mean(d)

ci_low, ci_high = np.quantile(
    boot_means,
    [0.025, 0.975]
)

print("Mean paired MSE improvement:", observed_mean)
print("95% bootstrap CI:", ci_low, ci_high)

Mean paired MSE improvement: -0.07865029206024991
95% bootstrap CI: -0.28556098426192145 0.1050473894061008


In [87]:
results = []

for region in sorted(paired["Region"].unique()):
    for horizon in sorted(paired["Horizon"].unique()):

        sub = (
            paired[
                (paired["Region"] == region) &
                (paired["Horizon"] == horizon)
            ]
            .sort_values("Forecast_date")
            .dropna(subset=["d_t"])
        )

        d = sub["d_t"].to_numpy()

        boot = block_bootstrap_mean_difference(
            d,
            block_length=8,
            n_boot=5000,
            seed=42
        )

        mean_diff = d.mean()
        ci_low, ci_high = np.quantile(
            boot,
            [0.025, 0.975]
        )

        results.append({
            "Region": region,
            "Horizon": horizon,
            "N": len(d),
            "Mean_MSE_diff": mean_diff,
            "CI_lower": ci_low,
            "CI_upper": ci_high,
            "RF_better": ci_low > 0,
            "Climatology_better": ci_high < 0
        })

bootstrap_results = pd.DataFrame(results)

In [89]:
bootstrap_results.to_csv(Path(PROJECT_ROOT / "outputs"/ "predictions_2021_2024_RF" / "bootstrap_results_EESN.csv")) 

# MSE decomposition 

In [91]:
def murphy_mse_decomposition(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    o = y_true[mask]
    f = y_pred[mask]

    mu_o = np.mean(o)
    mu_f = np.mean(f)

    # ddof=0 keeps the identity consistent with mean squared error
    sigma_o = np.std(o, ddof=0)
    sigma_f = np.std(f, ddof=0)

    rho = np.corrcoef(o, f)[0, 1]

    correlation_component = sigma_o**2 * (1 - rho**2)
    conditional_bias = (sigma_f - rho * sigma_o)**2
    unconditional_bias = (mu_f - mu_o)**2

    mse = np.mean((f - o)**2)

    return {
        "MSE": mse,
        "Correlation_component": correlation_component,
        "Conditional_bias": conditional_bias,
        "Unconditional_bias": unconditional_bias,
        "rho": rho,
        "forecast_sd": sigma_f,
        "observed_sd": sigma_o,
        "sum_components": (
            correlation_component
            + conditional_bias
            + unconditional_bias
        )
    }


In [92]:
results = []

for region in sorted(eesn["Region"].unique()):
    for horizon in sorted(eesn["Horizon"].unique()):

        sub = (
            eesn[
                (eesn["Region"] == region) &
                (eesn["Horizon"] == horizon)
            ]
            .sort_values("Forecast_date")
            .dropna(subset=["Obs_detrended"])
            .dropna(subset=["Pred_detrended"])
        )

        y_true = sub["Obs_detrended"].to_numpy()
        y_pred = sub["Pred_detrended"].to_numpy()

        res = murphy_mse_decomposition(y_true, y_pred)

        # Add Region and Horizon to the returned metrics
        res["Region"] = region
        res["Horizon"] = horizon

        results.append(res)

cols = ["Region", "Horizon", "MSE", "Correlation_component", "Conditional_bias",
        "Unconditional_bias", "rho", "forecast_sd", "observed_sd", "sum_components"]

MSE_decomp_results = pd.DataFrame(results)[cols]
MSE_decomp_results

,Region,Horizon,MSE,Correlation_component,Conditional_bias,Unconditional_bias,rho,forecast_sd,observed_sd,sum_components
0,MW,1,7.137406,6.657328,0.021858,0.458220,0.570863,1.941813,3.142556,7.137406
1,MW,2,10.563302,9.681171,0.260491,0.621640,0.140333,0.951387,3.142556,10.563302
2,MW,3,11.217567,9.715971,1.136674,0.364922,-0.127160,0.666543,3.142556,11.217567
3,MW,4,11.027298,9.874322,0.434636,0.718340,-0.011620,0.622754,3.142556,11.027298
4,NE,1,4.925272,4.497176,0.364020,0.064076,0.452365,1.679002,2.377861,4.925272
5,NE,2,6.044006,5.616288,0.285428,0.142290,0.081910,0.729026,2.377861,6.044006
6,NE,3,6.389202,5.651523,0.436161,0.301518,-0.021856,0.608454,2.377861,6.389202
7,NE,4,6.313448,5.651643,0.419037,0.242767,-0.021365,0.596528,2.377861,6.313448
8,SE,1,4.211344,3.976083,0.012447,0.222814,0.478432,1.197971,2.270762,4.211344
9,SE,2,5.023551,4.873784,0.005454,0.144313,0.234097,0.605430,2.270762,5.023551


In [36]:
results = []

for region in sorted(RF_preds["Region"].unique()):
    for horizon in sorted(RF_preds["Horizon"].unique()):

        sub = (
            RF_preds[
                (RF_preds["Region"] == region) &
                (RF_preds["Horizon"] == horizon)
            ]
            .sort_values("Forecast_date")
        )

        y_true = sub["Obs_detrended"].to_numpy()
        y_pred = sub["Pred_detrended"].to_numpy()

        res = murphy_mse_decomposition(y_true, y_pred)

        # Add Region and Horizon to the returned metrics
        res["Region"] = region
        res["Horizon"] = horizon

        results.append(res)

cols = ["Region", "Horizon", "MSE", "Correlation_component", "Conditional_bias",
        "Unconditional_bias", "rho", "forecast_sd", "observed_sd", "sum_components"]

MSE_decomp_results = pd.DataFrame(results)[cols]

In [39]:
display(MSE_decomp_results.style.hide(axis="index"))

Region,Horizon,MSE,Correlation_component,Conditional_bias,Unconditional_bias,rho,forecast_sd,observed_sd,sum_components
MW,1,7.229086,6.835284,0.074585,0.319216,0.523704,1.334119,3.068949,7.229086
MW,2,9.016530,8.334826,0.045537,0.636167,0.114296,0.545546,2.906054,9.016530
MW,3,9.535913,8.462838,0.285480,0.787595,-0.050935,0.385936,2.912877,9.535913
MW,4,9.373120,8.512855,0.084524,0.775741,0.027842,0.371994,2.918811,9.373120
NE,1,4.664890,4.510286,0.000732,0.153872,0.454104,1.109498,2.383687,4.664890
NE,2,5.853556,5.509346,0.022045,0.322165,0.119511,0.431019,2.364144,5.853556
NE,3,5.858169,5.553395,0.012939,0.291835,0.100594,0.352016,2.368579,5.858169
NE,4,6.080028,5.571269,0.010470,0.498289,0.101992,0.344322,2.372727,6.080028
SE,1,3.842245,3.734919,0.034489,0.072837,0.528728,1.018137,2.276878,3.842245
SE,2,5.118706,4.874250,0.022629,0.221826,0.131313,0.442871,2.227055,5.118706


In [41]:
MSE_decomp_results.to_latex(
        index=False,
        escape=False,
        longtable=False,
        caption="MSE decomposition for RF predictions.",
        label="tab:MSE_decomp",
        float_format="%.3f"
    )

'\\begin{table}\n\\caption{MSE decomposition for RF predictions.}\n\\label{tab:MSE_decomp}\n\\begin{tabular}{lrrrrrrrrr}\n\\toprule\nRegion & Horizon & MSE & Correlation_component & Conditional_bias & Unconditional_bias & rho & forecast_sd & observed_sd & sum_components \\\\\n\\midrule\nMW & 1 & 7.229 & 6.835 & 0.075 & 0.319 & 0.524 & 1.334 & 3.069 & 7.229 \\\\\nMW & 2 & 9.017 & 8.335 & 0.046 & 0.636 & 0.114 & 0.546 & 2.906 & 9.017 \\\\\nMW & 3 & 9.536 & 8.463 & 0.285 & 0.788 & -0.051 & 0.386 & 2.913 & 9.536 \\\\\nMW & 4 & 9.373 & 8.513 & 0.085 & 0.776 & 0.028 & 0.372 & 2.919 & 9.373 \\\\\nNE & 1 & 4.665 & 4.510 & 0.001 & 0.154 & 0.454 & 1.109 & 2.384 & 4.665 \\\\\nNE & 2 & 5.854 & 5.509 & 0.022 & 0.322 & 0.120 & 0.431 & 2.364 & 5.854 \\\\\nNE & 3 & 5.858 & 5.553 & 0.013 & 0.292 & 0.101 & 0.352 & 2.369 & 5.858 \\\\\nNE & 4 & 6.080 & 5.571 & 0.010 & 0.498 & 0.102 & 0.344 & 2.373 & 6.080 \\\\\nSE & 1 & 3.842 & 3.735 & 0.034 & 0.073 & 0.529 & 1.018 & 2.277 & 3.842 \\\\\nSE & 2 & 5.119 & 4

In [68]:
eesn_preds = pd.read_csv(Path(PROJECT_ROOT / "notebooks"/ "test_pred_obs.csv"))
eesn=eesn_preds[
    (eesn_preds["Region"] == "W") &
    (eesn_preds["Horizon"] == 1)
]

np.sqrt(
    np.mean(
        (eesn["Pred_detrended"] - eesn["Obs_detrended"])**2
    )
)

2.2428757085091777

In [69]:
paired_EESN = pd.read_csv("paired_EESN.csv")

In [70]:
check = (
    paired_EESN
    .dropna(subset=["RF_SE"])
    .groupby(["Region", "Horizon"])
    .agg(
        EESN_MSE=("RF_SE", "mean"),
        Clim_MSE=("Clim_SE", "mean")
    )
)

check["EESN_RMSE"] = np.sqrt(check["EESN_MSE"])
check["Clim_RMSE"] = np.sqrt(check["Clim_MSE"])

print(check)

                EESN_MSE   Clim_MSE  EESN_RMSE  Clim_RMSE
Region Horizon                                           
MW     1        3.791000  10.547417   1.947049   3.247679
       2        0.906108  10.547417   0.951897   3.247679
       3        0.490730  10.547417   0.700521   3.247679
       4        0.388603  10.547417   0.623380   3.247679
NE     1        2.914654   5.970444   1.707236   2.443449
       2        0.565749   5.970444   0.752163   2.443449
       3        0.370392   5.970444   0.608598   2.443449
       4        0.360693   5.970444   0.600577   2.443449
SE     1        1.437082   5.422781   1.198784   2.328687
       2        0.385116   5.422781   0.620577   2.328687
       3        0.295548   5.422781   0.543643   2.328687
       4        0.290755   5.422781   0.539217   2.328687
SW     1        0.531769   5.050618   0.729225   2.247358
       2        0.166890   5.050618   0.408521   2.247358
       3        0.130368   5.050618   0.361066   2.247358
       4      

In [75]:
paired.head()

,Forecast_date,Observed,Predicted,Obs_detrended,Pred_detrended,Horizon,Region,Z_score,EESN_SE,Clim_SE,d_t
0,202101,271.973366,273.557932,3.506021,1.584566,1,MW,0.643581,3.691993,12.292187,8.600194
1,202102,272.174428,275.998336,3.991220,3.823908,1,MW,0.836872,0.027993,15.929837,15.901844
2,202103,271.035402,274.117991,2.877007,3.082589,1,MW,0.649016,0.042264,8.277170,8.234906
3,202104,266.311598,265.878092,-2.080605,-0.433505,1,MW,-0.579591,2.712935,4.328915,1.615980
4,202105,270.436346,270.433540,1.561993,-0.002806,1,MW,0.255057,2.448597,2.439823,-0.008774


In [76]:
paired_EESN.head()

,Forecast_date,Region,Horizon,RF_pred,Observed_RF,Clim_pred,Observed_external,obs_diff,Observed,RF_SE,Clim_SE,d_t
0,202101,MW,1,273.557932,271.973366,268.467345,271.973366,0.0,271.973366,2.510848,12.292187,9.781339
1,202102,MW,1,275.998336,272.174428,268.183208,272.174428,0.0,272.174428,14.622271,15.929837,1.307566
2,202103,MW,1,274.117991,271.035402,268.158395,271.035402,0.0,271.035402,9.502356,8.277170,-1.225186
3,202104,MW,1,265.878092,266.311598,268.392202,266.311598,0.0,266.311598,0.187927,4.328915,4.140988
4,202105,MW,1,270.433540,270.436346,268.874352,270.436346,0.0,270.436346,0.000008,2.439823,2.439815


In [93]:
 0.136 + .397



0.533